# Transparent GPU-Hour Allocation

This notebook runs Team A's exact six-team comparison: random-order FCFS versus a carbon-aware VCG priority mechanism.

For each team, estimated emissions are $e_i = 0.24d_i$ and the selection score is $v_i - 0.5e_i$. The mechanism chooses the feasible group with the highest total score, subject to 100 GPU-hours.


## Game design and solution benchmark

**Game class.** This is a static, simultaneous-move game with incomplete information. Each team privately knows its project value $v_i$ and simultaneously submits a report $r_i$. GPU demand $d_i$ and estimated emissions $e_i$ are observable.

**Auction format.** Our proposed rule is a sealed-bid, carbon-aware multi-team VCG direct mechanism: a generalization of second-price/Vickrey logic from one winner to a feasible group of teams. FCFS is a non-strategic baseline, not the auction game.

**Solution concept.** The benchmark is dominant-strategy incentive compatibility (DSIC): reporting $r_i=v_i$ is optimal for each team regardless of the other teams' reports, provided priority credits have a meaningful opportunity cost. DSIC is stronger than Bayesian Nash equilibrium (BNE), so truthful reporting is also a BNE.


## Information structure: Harsanyi-style direct mechanism

This compact diagram represents the full six-team model. It is not a sequential auction: the reports are submitted at the same time.

```text
Nature assigns each team a private project value v_i
                   |
Each team observes only its own value v_i
                   |
All six teams simultaneously submit reports r_i
                   |
Mechanism observes reports r_i, demands d_i, and emissions e_i
                   |
Choose feasible S that maximizes sum_{i in S} (r_i - 0.5e_i)
subject to sum_{i in S} d_i <= 100
                   |
Allocate GPU-hours to S and charge VCG externality credits
```

The code below implements this allocation for six teams by checking all $2^6=64$ possible groups.


In [ ]:
# Make the solver available both locally and when this notebook is opened in Google Colab.
from pathlib import Path
import os
import subprocess
import sys

repository_url = 'https://github.com/GihoonE/COMPSCI206-PS2.git'
repository_name = 'COMPSCI206-PS2'
project_root = Path.cwd()

if not (project_root / 'src' / 'gpu_allocation.py').exists():
    clone_path = project_root / repository_name
    if not clone_path.exists():
        subprocess.run(['git', 'clone', repository_url, repository_name], check=True)
    os.chdir(clone_path)
    project_root = Path.cwd()

assert (project_root / 'src' / 'gpu_allocation.py').exists()
sys.path.insert(0, str(project_root))

from src.gpu_allocation import (
    CAPACITY_GPU_HOURS,
    CARBON_PENALTY_PER_KG_CO2E,
    EMISSIONS_KG_CO2E_PER_GPU_HOUR,
    PRIORITY_CREDITS_PER_SCORE_UNIT,
    default_example,
    fcfs_allocation,
    outcome_metrics,
    team_rows,
    vcg_allocation,
)


## Model inputs and exact algorithm

The values below are the announced baseline parameters. The code has no random draws: the FCFS order and all six team inputs are fixed, so another reader should reproduce the same output.

**Language-independent pseudocode**

1. Read each team's demand $d_i$, private value report $r_i$, and observable emissions $e_i=0.24d_i$.
2. For FCFS, serve each full request in the fixed public arrival order when it fits in remaining capacity.
3. For VCG, enumerate all 64 possible groups of six teams; remove groups whose total demand exceeds 100.
4. Select the feasible group with the largest total score $\sum_{i\in S}(r_i-\lambda e_i)$.
5. For every selected team, remove that team, solve again, and charge its VCG externality in priority credits.
6. Report allocation, payments, quasi-linear utility, project value, carbon-adjusted score, GPU use, unused capacity, and emissions.


In [ ]:
parameters = {
    'teams': 6,
    'capacity_gpu_hours': CAPACITY_GPU_HOURS,
    'emissions_kg_co2e_per_gpu_hour': EMISSIONS_KG_CO2E_PER_GPU_HOUR,
    'baseline_carbon_penalty_lambda': CARBON_PENALTY_PER_KG_CO2E,
    'priority_credits_per_score_unit': PRIORITY_CREDITS_PER_SCORE_UNIT,
    'FCFS_arrival_order': 'A -> B -> E -> C -> D -> F',
}
for name, value in parameters.items():
    print(f'{name}: {value}')


In [ ]:
# The six teams use the project's Low / Medium / High values: 3, 6, and 9.
teams, arrival_order = default_example()
fcfs = fcfs_allocation(teams, arrival_order, CAPACITY_GPU_HOURS)
vcg = vcg_allocation(teams, CAPACITY_GPU_HOURS)

fcfs_summary = outcome_metrics(teams, fcfs)
vcg_summary = outcome_metrics(teams, vcg)

print('FCFS arrival order:', ' -> '.join(teams[i].name for i in arrival_order))
print('FCFS selected teams:', fcfs_summary['selected_teams'])
print('VCG selected teams:', vcg_summary['selected_teams'])


In [ ]:
# Compact comparison used in the paper, poster, and Hugging Face audit.
for label, summary in [('FCFS', fcfs_summary), ('Carbon-aware VCG', vcg_summary)]:
    print(f'\n{label}')
    for key in ['teams_served', 'gpu_hours_used', 'unused_gpu_hours',
                'total_true_project_value', 'carbon_adjusted_true_score',
                'total_estimated_emissions_kg_co2e',
                'total_priority_payment_credits']:
        print(f'  {key}: {summary[key]}')


In [ ]:
# Compact actual-output table for the paper, poster, and Hugging Face audit.
metrics = [
    'teams_served', 'gpu_hours_used', 'unused_gpu_hours',
    'total_true_project_value', 'carbon_adjusted_true_score',
    'total_estimated_emissions_kg_co2e', 'total_priority_payment_credits',
]
print(f"{'metric':38} {'FCFS':>12} {'carbon-aware VCG':>20}")
print('-' * 74)
for metric in metrics:
    print(f"{metric:38} {str(fcfs_summary[metric]):>12} {str(vcg_summary[metric]):>20}")


In [ ]:
# VCG payment and utility audit: utility is true project value minus payment in score units.
print(f"{'team':8} {'selected':9} {'payment credits':17} {'utility score units':21}")
print('-' * 64)
for row in team_rows(teams, vcg):
    print(f"{row['team']:8} {str(row['selected']):9} {row['priority_payment_credits']:17.1f} {row['quasi_linear_utility_score_units']:21.1f}")


## Carbon-penalty sensitivity check

The baseline uses $\lambda=0.5$. This check changes only the announced carbon-penalty weight, while holding demands, values, capacity, and the allocation algorithm fixed. It is a model robustness check, not new behavioral evidence.


In [ ]:
for penalty in (0.0, 0.5, 1.0):
    sensitivity_outcome = vcg_allocation(
        teams, CAPACITY_GPU_HOURS, carbon_penalty_per_kg_co2e=penalty
    )
    sensitivity = outcome_metrics(teams, sensitivity_outcome)
    print(
        f"lambda={penalty:.1f} | selected: {sensitivity['selected_teams']} | "
        f"GPU-hours: {sensitivity['gpu_hours_used']} | "
        f"emissions: {sensitivity['total_estimated_emissions_kg_co2e']} | "
        f"score: {sensitivity['carbon_adjusted_true_score']}"
    )


In [ ]:
# Fresh-run verification for the fixed baseline.
assert fcfs_summary['gpu_hours_used'] == 95.0
assert vcg_summary['gpu_hours_used'] == 90.0
assert vcg_summary['total_true_project_value'] == 30.0
assert vcg_summary['total_priority_payment_credits'] == 96.0
print('Fresh-run verification passed: FCFS and VCG baseline outputs match the saved audit.')


In [ ]:
# Transparent allocation audit: one row for every team under the VCG rule.
for row in team_rows(teams, vcg):
    print(row)

# VCG priority credits are the externality each selected team creates for others.
# The separate run_simulation.py file writes these same results to CSV files.
